In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

import folium
import folium.plugins

from folium.plugins import TimeSliderChoropleth

import branca.colormap #utility module for dealing with colourmaps
from branca.colormap import linear
from branca.colormap import StepColormap 

#import pydeck as pdk

import h3pandas
from datetime import datetime
import osmnx as ox

#Import trig stuff from math
from math import sin, cos, sqrt, atan2,radians

pd.set_option('display.max_columns', None) ### This line makes all the columns display, rather than ellipses shorten
pd.options.mode.chained_assignment = None  # default='warn'

In [2]:
parcel_and_tax_frame = gpd.read_file("C:/Users/aaron/Fun Mapping Work/Data/Value_Capture_Study/PARCEL_ADDRESS_PUB_AREA_3069_4362620268901785570", columns = ('geometry', 'ACCNT_NUM', 'LAT', 'LON'))

#parcel_and_tax_frame = parcel_and_tax_frame.set_crs('epsg:4326', allow_override=True)


parcel_and_tax_frame['ACCNT_NUM'] = pd.to_numeric(parcel_and_tax_frame['ACCNT_NUM'], errors= 'coerce')

C:\Users\aaron\AppData\Roaming\Python\Python314\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:/Users/aaron/Fun Mapping Work/Data/Value_Capture_Study/PARCEL_ADDRESS_PUB_AREA_3069_4362620268901785570/Parcels_for_King_County_with_Address,_Property_and_Ownership_Information.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [3]:
old_data = pd.read_csv("C:/Users/aaron/Fun Mapping Work/Data/Value_Capture_Study/Maybe Older data/Tax Data/EXTR_TaxAcctReceivable_V.csv")

#data_2018 = old_data[old_data['BillYr'] == 2018]
#data_2027 = old_data[old_data['BillYr'] == 2027]

In [4]:
#combined_frame_2018 = pd.merge(left = parcel_and_tax_frame[['ADDR_FULL', 'ACCNT_NUM', 'geometry', 'LAT', 'LON', 'LOTSQFT', 'KCTP_CITY', 'PROP_NAME']], right = data_2018, left_on='ACCNT_NUM', right_on='AcctNbr')
#combined_frame_2027 = pd.merge(left = parcel_and_tax_frame[['ADDR_FULL', 'ACCNT_NUM', 'geometry', 'LAT', 'LON', 'LOTSQFT', 'KCTP_CITY', 'PROP_NAME']], right = data_2027, left_on='ACCNT_NUM', right_on='AcctNbr')
combined_frame_all = pd.merge(left = parcel_and_tax_frame[['ACCNT_NUM', 'geometry', 'LAT', 'LON']], right = old_data, left_on='ACCNT_NUM', right_on='AcctNbr')

In [5]:
trimmed_all_year_frame = combined_frame_all[['LAT', 'LON', 'ApprLandVal', 'geometry', 'BillYr']]

In [6]:
#combined_frame_2027['Land_Value_Per_SQFT'] = combined_frame_2027['ApprLandVal'] / combined_frame_2027['LOTSQFT']
#combined_frame_2018['Land_Value_Per_SQFT'] = combined_frame_2018['ApprLandVal'] / combined_frame_2018['LOTSQFT']
#combined_frame_all['Land_Value_Per_SQFT'] = combined_frame_all['ApprLandVal'] / combined_frame_all['LOTSQFT']

In [7]:
### get rid of lat/longs with zero
#null_rows_2018 = combined_frame_2018[combined_frame_2018["LAT"] < 30].index
#combined_frame_2018.drop(null_rows_2018, inplace=True)

#null_rows_2027 = combined_frame_2027[combined_frame_2027["LAT"] < 30].index
#combined_frame_2027.drop(null_rows_2027, inplace=True)

null_rows_all = trimmed_all_year_frame[trimmed_all_year_frame["LAT"] < 30].index
trimmed_all_year_frame.drop(null_rows_all, inplace=True)


In [8]:
seattle_city_place = "Seattle, Washington, USA"

### includes some of the water area, but overall correct
seattle_boundary = ox.geocode_to_gdf(seattle_city_place)
#seattle_boundary.explore()

In [9]:
trimmed_all_year_frame['BillYr'].info()

<class 'pandas.core.series.Series'>
Index: 17123861 entries, 2 to 18799471
Series name: BillYr
Non-Null Count     Dtype
--------------     -----
17123861 non-null  int64
dtypes: int64(1)
memory usage: 261.3 MB


In [10]:
##drop all rows before the year 2008
trimmed_all_year_frame = trimmed_all_year_frame[trimmed_all_year_frame['BillYr'] > 2008]

In [11]:
trimmed_all_year_frame.head()

,LAT,LON,ApprLandVal,geometry,BillYr
15,47.656178,-122.355756,314000,"POLYGON ((1265280.194 243031.563, 1265240.197 ...",2009
16,47.656178,-122.355756,266000,"POLYGON ((1265280.194 243031.563, 1265240.197 ...",2010
17,47.656178,-122.355756,266000,"POLYGON ((1265280.194 243031.563, 1265240.197 ...",2011
18,47.656178,-122.355756,266000,"POLYGON ((1265280.194 243031.563, 1265240.197 ...",2012
19,47.656178,-122.355756,266000,"POLYGON ((1265280.194 243031.563, 1265240.197 ...",2013


In [13]:
## trim by eliminating lats and longs 
#trimmed_all_year_frame = trimmed_all_year_frame[trimmed_all_year_frame['LAT']]


In [14]:
trimmed_all_year_frame.shape

(10583827, 5)

In [16]:
#Seattle_sized_2027_frame = gpd.clip(combined_frame_2027_updated_epsg, seattle_boundary, keep_geom_type=False, sort=False)
#Seattle_sized_2018_frame = gpd.clip(combined_frame_2018_updated_epsg, seattle_boundary, keep_geom_type=False, sort=False)
Seattle_sized_trimmed_all_year_frame = gpd.clip(trimmed_all_year_frame, seattle_boundary, keep_geom_type=False, sort=False)


C:\Users\aaron\AppData\Local\Temp\ipykernel_32720\369794856.py:3: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: PROJCS["NAD83(HARN) / Washington North (ftUS)",GEO ...
Right CRS: EPSG:4326

  Seattle_sized_trimmed_all_year_frame = gpd.clip(trimmed_all_year_frame, seattle_boundary, keep_geom_type=False, sort=False)


In [17]:
#seattle_boundary_updated_crs = seattle_boundary.to_crs(4957)

#combined_frame_2027_updated_epsg = combined_frame_2027.to_crs(4326)
#combined_frame_2018_updated_epsg = combined_frame_2018.to_crs(4326)
trimmed_all_year_frame_updated_epsg = trimmed_all_year_frame.to_crs(4326)

In [18]:
dfDict = { i: Seattle_sized_trimmed_all_year_frame[Seattle_sized_trimmed_all_year_frame['BillYr'] == i] for i in range(2010,2025) }

In [20]:
dfDict.get(2010)

,LAT,LON,ApprLandVal,geometry,BillYr


In [ ]:
trimmed_2027_Frame = Seattle_sized_2027_frame[['LAT', 'LON', 'ApprLandVal']]
trimmed_2018_Frame = Seattle_sized_2018_frame[['LAT', 'LON', 'ApprLandVal']]

trimmed_all_year_frame = Seattle_sized_all_year_frame[['LAT', 'LON', 'ApprLandVal']]

In [ ]:
trimmed_2018_Frame.head(1)

In [ ]:
hexagons_2018 = trimmed_2018_Frame.h3.geo_to_h3_aggregate(resolution=8, lat_col='LAT', lng_col='LON')
hexagons_2027 = trimmed_2027_Frame.h3.geo_to_h3_aggregate(resolution=8, lat_col='LAT', lng_col='LON')

In [ ]:
most_valuable_hexagon = hexagons_2027['ApprLandVal'].max()

In [ ]:
#hexagons_2027.explore(column = 'ApprLandVal', vmax = most_valuable_hexagon * 0.6)

In [ ]:
### create the style dictionary

def create_style_dict(frame_1, frame_2):
    style_dict = {}
    cmap = plt.get_cmap('viridis')
    opct = 0.7
    agg_max = 2000000 #this number doesnt actually have to be the max, just the max for the colormap

In [ ]:
## this line 'flattens' the dataframe. 
hexagons_2018 = hexagons_2018.reset_index()

In [ ]:
frameList = [hexagons_2018, hexagons_2027]

In [ ]:
#import branca.colormap #utility module for dealing with colourmaps
#from branca.colormap import linear
#from branca.colormap import StepColormap 

In [ ]:
olympic_colormap = branca.colormap.StepColormap(
    colors=['darkblue', 'blue', 'pink', 'red'],
    index=[0, 25, 50, 75, 100],
    vmin=0, vmax=100)

olympic_colormap.caption = "Total land value per hexagon"
olympic_colormap

In [ ]:
def create_style_dict(df_list):
#def create_style_dict(frame_2010, frame_2011):
    style_dict = {}
    cmap = plt.get_cmap('viridis')
    opct = 0.7
    agg_max = 2000000 #this number doesnt actually have to be the max, just the max for the colormap
    
    #for hex in all_hexagons_series:
        #style_dict[hex] = {}
        

    for frame_name, frame_content in df_list.items():
        epoch_time = int(datetime.datetime(int(frame_name[9:]), 1, 1).timestamp())
        grouped_frame = frame_content.groupby('h3_09').agg({'EstProjectCost':'sum'}).reset_index()
        
        for hex_index, hex_row in grouped_frame.iterrows():
            #tracker = tracker + 1000000
            current_hex_label = hex_row['h3_09']
            current_hex_agg_cost = hex_row['EstProjectCost']
            dict_entry = {epoch_time: {'proj_cost': current_hex_agg_cost, 'opacity': opct, 'color': get_color(cmap, current_hex_agg_cost, 0, agg_max)}}
        
            if (style_dict[current_hex_label]):
                style_dict[current_hex_label] = style_dict[current_hex_label] | dict_entry
            else:
                style_dict[current_hex_label] = dict_entry
    
    return style_dict
    
style_dict = create_style_dict(frameList)

In [ ]:
def hexagons_dataframe_to_geojson(df_hex, file_output = None, column_name = "value"):
    
    ## Produce the GeoJSON for a dataframe, constructing the geometry from the "h3_09" column
    ## and with a property matching the one in column_name
       
    list_features = []
    
    for i,row in df_hex.iterrows():
        try:
            geometry_for_row = { "type" : "Polygon", "coordinates": [h3.h3_to_geo_boundary(h=row["h3_08"],geo_json=True)]}
            print (type(geometry_for_row))
            feature = Feature(geometry = geometry_for_row , id=row["h3_08"], properties = {column_name : row[column_name]})
           # print (feature)
            list_features.append(feature)
        except:
            print("An exception occurred for hex " + row["h3_08"])             

    feat_collection = FeatureCollection(list_features)
    geojson_result = json.dumps(feat_collection)
    return geojson_result

In [ ]:
hexagons_2018_geojsond = hexagons_dataframe_to_geojson(hexagons_2018, column_name='ApprLandVal')

In [ ]:
styledict = {}

In [ ]:
folium_map = folium.Map([0, 0], zoom_start=2)

TimeSliderChoropleth(
    hexagons_2018.to_json(),
    styledict=styledict,
    overlay=False,
    highlight=True
).add_to(folium_map)

folium.Choropleth(
    geo_data = geojson_data,
    data = df_aggreg,
    #tooltip = tt,
    fill_opacity=0.0,
    fill_color= 'viridis',
    columns = ['h3_09', 'EstProjectCost'],
    key_on='feature.id',
    highlight=True,
    legend_name='Estimated Project Cost by Year',
).add_to(seattle_base_map)
   # folium.LayerControl.add_to(seattle_base_map)

In [ ]:
folium_map